# Parte 1 — Dataset EscutIA: preparação, validação e divisão

> Este notebook é executado inteiramente dentro de `EscutIA/dataset`. A fonte anterior não faz parte deste fluxo e não precisa existir para esta aula.

Vamos preparar os dados para uma capacidade especializada de classificação de sentimentos. O modelo deverá receber um texto e responder somente com um JSON válido no formato `{"sentimento":"positivo"}`. O notebook não executa fine-tuning.

Vamos trabalhar em 11 passos. Execute as células na ordem e confira o resultado antes de continuar.

## Antes de começar

A pasta `dataset` já contém `dados/dataset_local.csv`, os scripts, os notebooks e as pastas de saída. Ao executar as células, o notebook baixa e unifica a fonte automaticamente usando a revisão fixada.

Na primeira execução, mantenha `EXECUTAR_DOWNLOAD = True` e `LIMPAR_RESULTADOS_ANTERIORES = True`. A limpeza remove somente resultados gerados dentro do dataset; as fontes locais permanecem preservadas.

In [15]:
EXECUTAR_DOWNLOAD = True
LIMPAR_RESULTADOS_ANTERIORES = True

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import re
import shutil
import subprocess
import sys
import unicodedata

import pandas as pd
from IPython.display import display
from jsonschema import Draft202012Validator
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'dataset' and (BASE_DIR / 'dataset').is_dir():
    BASE_DIR = BASE_DIR / 'dataset'
if BASE_DIR.name != 'dataset':
    raise RuntimeError('Abra este notebook com o diretório de trabalho em EscutIA/dataset.')

DATA_DIR = BASE_DIR / 'dados'
LOCAL_DATASET = DATA_DIR / 'dataset_local.csv'
DOWNLOAD_SCRIPT = BASE_DIR / 'scripts' / 'baixar_dataset.py'
DATASET_UNIFICADO = DATA_DIR / 'dataset.csv'
TRABALHO = DATA_DIR / 'trabalho'
PREPARADOS = DATA_DIR / 'preparados'
RELATORIOS = DATA_DIR / 'relatorios'
ROTULOS = {'negativo', 'neutro', 'positivo'}

for caminho in [LOCAL_DATASET, DOWNLOAD_SCRIPT]:
    if not caminho.exists():
        raise FileNotFoundError(f'Arquivo necessário não encontrado dentro do dataset: {caminho}')

for pasta in [TRABALHO, PREPARADOS, RELATORIOS]:
    pasta.mkdir(parents=True, exist_ok=True)
    if LIMPAR_RESULTADOS_ANTERIORES:
        for item in pasta.iterdir():
            if item.name == '.gitkeep':
                continue
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()

MANIFESTO = BASE_DIR / 'manifesto_dataset.json'
if LIMPAR_RESULTADOS_ANTERIORES and MANIFESTO.exists():
    MANIFESTO.unlink()

print(f'Pasta da aula: {BASE_DIR}')
print(f'Fontes e artefatos da aula: {DATA_DIR}')
print('Resultados anteriores limpos: dados/trabalho, dados/preparados e dados/relatorios') if LIMPAR_RESULTADOS_ANTERIORES else None

Pasta da aula: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\dataset
Fontes e artefatos da aula: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\dataset\dados
Resultados anteriores limpos: dados/trabalho, dados/preparados e dados/relatorios


## Passo 1 — Baixar e unificar o dataset

Vamos usar o script `scripts/baixar_dataset.py`. Ele baixa os splits `train`, `validation` e `test` do dataset de sentimentos em português, converte os rótulos para `negativo`, `neutro` e `positivo`, une os dados locais desta pasta e remove textos duplicados.

In [16]:
def sha256_arquivo(arquivo):
    digest = hashlib.sha256()
    with arquivo.open('rb') as stream:
        for bloco in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(bloco)
    return digest.hexdigest()

if not LOCAL_DATASET.exists():
    raise FileNotFoundError(f'Coloque o dataset local em: {LOCAL_DATASET}')
if EXECUTAR_DOWNLOAD:
    subprocess.run([sys.executable, str(DOWNLOAD_SCRIPT)], cwd=BASE_DIR, check=True)
elif not DATASET_UNIFICADO.exists():
    raise FileNotFoundError('O dataset unificado ainda não existe. Execute novamente com EXECUTAR_DOWNLOAD = True.')
else:
    print(f'Usando dataset unificado existente: {DATASET_UNIFICADO}')

df_original = pd.read_csv(DATASET_UNIFICADO, encoding='utf-8-sig')
print(f'Registros: {len(df_original)}')
print(f'Campos: {list(df_original.columns)}')
print(f'SHA-256 da fonte: {sha256_arquivo(DATASET_UNIFICADO)}')
display(df_original.head())
display(df_original.isna().sum().rename('valores_vazios').to_frame())
display(df_original['rotulo'].value_counts(dropna=False).rename('quantidade').to_frame())

Registros: 3063
Campos: ['id', 'texto', 'rotulo']
SHA-256 da fonte: 74c0c3b15ad86a6f54afb8fc9837d4e717f7894a4a9f85c5d58527705b70f81c


,id,texto,rotulo
0,1,Acordei animado e com vontade de começar o dia.,positivo
1,2,Foi muito bom conversar com meus amigos ontem.,positivo
2,3,Estou orgulhoso do resultado que consegui no t...,positivo
3,4,Hoje encontrei uma solução para um problema di...,positivo
4,5,Sinto esperança de que as coisas vão melhorar.,positivo


,valores_vazios
id,0
texto,0
rotulo,0


,quantidade
rotulo,
positivo,1021
neutro,1021
negativo,1021


In [17]:
relatorio_01 = {
    'passo': 1,
    'arquivo': str(DATASET_UNIFICADO),
    'dataset_local': str(LOCAL_DATASET),
    'script_unificacao': str(DOWNLOAD_SCRIPT),
    'sha256_dataset_local': sha256_arquivo(LOCAL_DATASET),
    'sha256': sha256_arquivo(DATASET_UNIFICADO),
    'registros': len(df_original),
    'campos': list(df_original.columns),
    'vazios': df_original.isna().sum().to_dict(),
    'distribuicao_rotulos': df_original['rotulo'].value_counts(dropna=False).to_dict(),
}
(RELATORIOS / '01_inspecao.json').write_text(json.dumps(relatorio_01, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Relatório de inspeção salvo.')

Relatório de inspeção salvo.


## Passo 2 — Definir o schema dos exemplos

Cada registro precisa ter um identificador, um texto e um rótulo permitido. Se esta célula falhar, pare, corrija a fonte ou a regra de preparação e repita a análise.

In [18]:
obrigatorios = {'id', 'texto', 'rotulo'}
problemas_schema = []
problemas_schema.extend(f'Campo ausente: {campo}' for campo in obrigatorios - set(df_original.columns))
if not problemas_schema:
    if df_original[list(obrigatorios)].isna().any().any():
        problemas_schema.append('Existe valor vazio em id, texto ou rotulo')
    if df_original['id'].astype(str).duplicated().any():
        problemas_schema.append('Existem ids duplicados')
    rotulos_encontrados = set(df_original['rotulo'].astype(str).str.strip().str.casefold())
    invalidos = rotulos_encontrados - ROTULOS
    if invalidos:
        problemas_schema.append(f'Rótulos inválidos: {sorted(invalidos)}')

status_schema = 'PASS' if not problemas_schema else 'BLOCKED'
print(status_schema)
display(pd.DataFrame({'problema': problemas_schema or ['Nenhum problema encontrado']}))
(RELATORIOS / '02_schema.json').write_text(json.dumps({'passo': 2, 'status': status_schema, 'problemas': problemas_schema}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert not problemas_schema, 'O schema está bloqueado. Corrija os dados antes de continuar.'

PASS


,problema
0,Nenhum problema encontrado


## Passo 3 — Organizar instrução, contexto e resposta

Agora deixamos explícito o que queremos que o modelo aprenda:

- `instruction`: o que o modelo deve fazer;
- `context`: o texto que será analisado;
- `response`: o rótulo usado para construir o JSON final.

In [19]:
INSTRUCAO = 'Classifique o sentimento predominante do texto como negativo, neutro ou positivo e responda somente com um JSON válido no formato {"sentimento":"<rotulo>"}.'

organizados = pd.DataFrame({
    'id': df_original['id'].astype(str).str.strip(),
    'instruction': INSTRUCAO,
    'context': df_original['texto'].astype(str),
    'response': df_original['rotulo'].astype(str).str.strip().str.casefold(),
})
display(organizados.head())
(TRABALHO / '03_organizados.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in organizados.to_dict(orient='records')), encoding='utf-8')

,id,instruction,context,response
0,1,Classifique o sentimento predominante do texto...,Acordei animado e com vontade de começar o dia.,positivo
1,2,Classifique o sentimento predominante do texto...,Foi muito bom conversar com meus amigos ontem.,positivo
2,3,Classifique o sentimento predominante do texto...,Estou orgulhoso do resultado que consegui no t...,positivo
3,4,Classifique o sentimento predominante do texto...,Hoje encontrei uma solução para um problema di...,positivo
4,5,Classifique o sentimento predominante do texto...,Sinto esperança de que as coisas vão melhorar.,positivo


933925

## Passo 4 — Limpar e normalizar os dados

Vamos corrigir somente formatação: espaços extras, Unicode e capitalização dos rótulos. Registros que não podem ser usados são preservados em uma lista de rejeitados com o motivo. Não vamos inventar conteúdo.

In [20]:
def normalizar_texto(valor):
    valor = unicodedata.normalize('NFKC', str(valor or ''))
    return re.sub(r'\s+', ' ', valor).strip()

limpos, rejeitados_04 = [], []
for registro in organizados.to_dict(orient='records'):
    novo = {**registro}
    novo['instruction'] = normalizar_texto(novo['instruction'])
    novo['context'] = normalizar_texto(novo['context'])
    novo['response'] = normalizar_texto(novo['response']).casefold()
    motivos = []
    if not novo['id']: motivos.append('id_vazio')
    if not novo['context']: motivos.append('contexto_vazio')
    if novo['response'] not in ROTULOS: motivos.append('rotulo_invalido')
    if motivos: rejeitados_04.append({'registro': novo, 'motivos': motivos})
    else: limpos.append(novo)

print(f'Mantidos: {len(limpos)} | Rejeitados: {len(rejeitados_04)}')
(TRABALHO / '04_limpos.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in limpos), encoding='utf-8')
(TRABALHO / '04_rejeitados.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in rejeitados_04), encoding='utf-8')
(RELATORIOS / '04_limpeza.json').write_text(json.dumps({'passo': 4, 'mantidos': len(limpos), 'rejeitados': len(rejeitados_04)}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

Mantidos: 3063 | Rejeitados: 0


56

## Passo 5 — Remover duplicidades e exemplos inconsistentes

Textos repetidos podem fazer o modelo decorar exemplos. Textos iguais com rótulos diferentes são conflitos que precisam de decisão humana. Vamos separar esses casos sem apagar a evidência.

In [21]:
def chave_comparacao(valor):
    return re.sub(r'[^\w\s]', '', normalizar_texto(valor).casefold(), flags=re.UNICODE)

grupos = {}
for registro in limpos:
    grupos.setdefault(chave_comparacao(registro['context']), []).append(registro)

sem_duplicatas, rejeitados_05 = [], []
for grupo in grupos.values():
    rotulos_do_grupo = {registro['response'] for registro in grupo}
    if len(rotulos_do_grupo) > 1:
        rejeitados_05.extend({'registro': registro, 'motivo': 'CONFLITO_DE_ROTULO'} for registro in grupo)
    else:
        sem_duplicatas.append(grupo[0])
        rejeitados_05.extend({'registro': registro, 'motivo': 'DUPLICATA_DE_TEXTO'} for registro in grupo[1:])

print(f'Mantidos: {len(sem_duplicatas)} | Separados para revisão: {len(rejeitados_05)}')
(TRABALHO / '05_sem_duplicatas.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in sem_duplicatas), encoding='utf-8')
(TRABALHO / '05_rejeitados.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in rejeitados_05), encoding='utf-8')
(RELATORIOS / '05_duplicidades.json').write_text(json.dumps({'passo': 5, 'mantidos': len(sem_duplicatas), 'rejeitados': len(rejeitados_05)}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

Mantidos: 3057 | Separados para revisão: 6


56

## Passo 6 — Verificar dados sensíveis, incorretos ou fora do domínio

Este detector é apenas uma triagem didática. Ele sinaliza e-mails, telefones, CPF e termos relacionados a crises, violência ou abuso. Um texto sensível pode ser relevante para a EscutIA, então não será excluído automaticamente.

In [22]:
PADROES = {
    'email': re.compile(r'\b[^\s@]+@[^\s@]+\.[^\s@]+\b'),
    'telefone': re.compile(r'(?:\+?55\s*)?(?:\(?\d{2}\)?\s*)?9?\d{4}[-\s]?\d{4}'),
    'cpf': re.compile(r'\b\d{3}[.\s]?\d{3}[.\s]?\d{3}[-\s]?\d{2}\b'),
    'termos_de_crise': re.compile(r'\b(suic[ií]d|autoagress[aã]o|me matar|viol[eê]ncia|abuso)\w*\b', re.IGNORECASE),
}

analisados = []
for registro in sem_duplicatas:
    texto = f"{registro['context']} {registro['response']}"
    alertas = [nome for nome, padrao in PADROES.items() if padrao.search(texto)]
    analisados.append({**registro, 'alertas': alertas, 'status_revisao': 'REVISAR' if alertas else 'OK'})

df_alertas = pd.DataFrame([r for r in analisados if r['status_revisao'] == 'REVISAR'])
print(f'Registros com alerta: {len(df_alertas)}')
display(df_alertas if not df_alertas.empty else pd.DataFrame({'resultado': ['Nenhum alerta encontrado no exemplo']}))
(TRABALHO / '06_conteudo_analisado.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in analisados), encoding='utf-8')
(RELATORIOS / '06_conteudo.json').write_text(json.dumps({'passo': 6, 'registros_com_alerta': len(df_alertas), 'alertas': df_alertas['alertas'].tolist() if not df_alertas.empty else []}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

Registros com alerta: 7


,id,instruction,context,response,alertas,status_revisao
0,209,Classifique o sentimento predominante do texto...,Por onde apresentamos a peça temos recebido vá...,neutro,[termos_de_crise],REVISAR
1,578,Classifique o sentimento predominante do texto...,"O desarmamento não acaba com a violência, assi...",neutro,[termos_de_crise],REVISAR
2,1214,Classifique o sentimento predominante do texto...,"Sexo, sangue, violência. Não é Game of Thrones...",neutro,[termos_de_crise],REVISAR
3,2429,Classifique o sentimento predominante do texto...,Querem evitar ser vítima de abuso. Mas e cm se...,neutro,[termos_de_crise],REVISAR
4,2600,Classifique o sentimento predominante do texto...,Já contei una 83392991 cortes de cabelo difere...,neutro,[telefone],REVISAR
5,2714,Classifique o sentimento predominante do texto...,#Encontro Na maioria das vezes a violencia est...,neutro,[termos_de_crise],REVISAR
6,2732,Classifique o sentimento predominante do texto...,Duplicou a violência no BR graças ao desarmame...,neutro,[termos_de_crise],REVISAR


317

### Checkpoint humano antes da divisão

Para cada registro com alerta, confira o texto e escolha uma ação:

- `M` — manter o registro;
- `R` — remover somente da cópia derivada;
- `T` — corrigir o texto na cópia derivada;
- `L` — corrigir o rótulo na cópia derivada.

O arquivo `dados/dataset.csv` nunca é alterado por esta revisão. Se não houver alertas, a célula segue automaticamente.

In [23]:
base_para_divisao = []
decisoes_revisao = []

for registro in analisados:
    if registro['status_revisao'] == 'OK':
        base_para_divisao.append(registro)
        continue

    print('\n' + '=' * 80)
    print(f"ID: {registro['id']}")
    print(f"Texto: {registro['context']}")
    print(f"Rótulo atual: {registro['response']}")
    print(f"Alertas: {', '.join(registro['alertas'])}")
    while True:
        decisao = input('Escolha M=manter, R=remover, T=corrigir texto ou L=corrigir rótulo: ').strip().upper()
        if decisao in {'M', 'R', 'T', 'L'}:
            break
        print('Opção inválida. Digite M, R, T ou L.')

    novo = {**registro}
    justificativa = input('Justificativa curta: ').strip()
    if decisao == 'R':
        novo['status_revisao'] = 'REMOVIDO_MANUALMENTE'
    elif decisao == 'T':
        novo['context'] = input('Digite o texto corrigido: ').strip()
        novo['status_revisao'] = 'CORRIGIDO_MANUALMENTE'
    elif decisao == 'L':
        while True:
            novo_rotulo = input('Digite o novo rótulo (negativo/neutro/positivo): ').strip().casefold()
            if novo_rotulo in ROTULOS:
                break
            print('Rótulo inválido.')
        novo['response'] = novo_rotulo
        novo['status_revisao'] = 'CORRIGIDO_MANUALMENTE'
    else:
        novo['status_revisao'] = 'APROVADO_MANUALMENTE'

    novo['decisao_revisao'] = decisao
    novo['justificativa_revisao'] = justificativa
    decisao_registro = {'id': novo['id'], 'decisao': decisao, 'justificativa': justificativa}
    if decisao == 'T': decisao_registro['replacement_text'] = novo['context']
    if decisao == 'L': decisao_registro['replacement_label'] = novo['response']
    decisoes_revisao.append(decisao_registro)
    if novo['status_revisao'] != 'REMOVIDO_MANUALMENTE':
        base_para_divisao.append(novo)

display(pd.DataFrame(decisoes_revisao) if decisoes_revisao else pd.DataFrame({'resultado': ['Nenhum alerta precisou de revisão manual.']}))
(TRABALHO / '06_revisados.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in base_para_divisao), encoding='utf-8')
(RELATORIOS / '06_revisao_manual.json').write_text(json.dumps({'passo': 6, 'decisoes': decisoes_revisao, 'mantidos_para_divisao': len(base_para_divisao)}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Registros liberados para a divisão: {len(base_para_divisao)}')


ID: 209
Texto: Por onde apresentamos a peça temos recebido vários relatos de abusos, pessoas se abrindo pela primeira vez #Encontro #HospitalDeBonecas
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 578
Texto: O desarmamento não acaba com a violência, assim como a proibição da maconha não acaba com o tráfico. Me julguem #TheNoite
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 1214
Texto: Sexo, sangue, violência. Não é Game of Thrones, é #MasterChefBR
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 2429
Texto: Querem evitar ser vítima de abuso. Mas e cm se evita q novos abusadores surjam? Ser um abusador tem uma causa. Temos q trata-la!! #Encontro
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 2600
Texto: Já contei una 83392991 cortes de cabelo diferente da Fátima 😂 #VideoShowAoVivo
Rótulo atual: neutro
Alertas: telefone

ID: 2714
Texto: #Encontro Na maioria das vezes a violencia esta camuflada dentro de casa
Rótulo atual: neutro
Alertas: termos_de_crise

ID: 2732
Texto: 

,id,decisao,justificativa
0,209,M,
1,578,M,
2,1214,M,
3,2429,M,
4,2600,M,
5,2714,M,
6,2732,M,


Registros liberados para a divisão: 3057


## Passo 7 — Separar treinamento, validação e avaliação

A divisão será estratificada para preservar a proporção dos três sentimentos. A avaliação ficará separada para ser usada somente na comparação final com o modelo base.

In [24]:
treino_validacao, avaliacao = train_test_split(
    base_para_divisao, test_size=0.20, random_state=42,
    stratify=[r['response'] for r in base_para_divisao],
)
treino, validacao = train_test_split(
    treino_validacao, test_size=0.25, random_state=42,
    stratify=[r['response'] for r in treino_validacao],
)

conjuntos = {'treino': treino, 'validacao': validacao, 'avaliacao': avaliacao}
resumo_divisao = pd.DataFrame({
    nome: pd.Series(Counter(registro['response'] for registro in registros))
    for nome, registros in conjuntos.items()
}).fillna(0).astype(int)
display(resumo_divisao)
for nome, registros in conjuntos.items():
    (TRABALHO / f'07_{nome}.jsonl').write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in registros), encoding='utf-8')
(RELATORIOS / '07_divisao.json').write_text(json.dumps({'passo': 7, 'seed': 42, 'quantidades': {nome: len(registros) for nome, registros in conjuntos.items()}, 'rotulos': resumo_divisao.to_dict()}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

,treino,validacao,avaliacao
negativo,612,204,204
neutro,610,204,204
positivo,611,204,204


410

## Passo 8 — Verificar vazamento entre os conjuntos

Nenhum texto normalizado pode aparecer em mais de um conjunto. Se houver sobreposição, o resultado da avaliação poderá parecer melhor do que realmente é.

In [25]:
chaves_conjuntos = {nome: {chave_comparacao(registro['context']) for registro in registros} for nome, registros in conjuntos.items()}
sobreposicoes = {}
for esquerda, direita in [('treino', 'validacao'), ('treino', 'avaliacao'), ('validacao', 'avaliacao')]:
    comuns = chaves_conjuntos[esquerda] & chaves_conjuntos[direita]
    sobreposicoes[f'{esquerda}_x_{direita}'] = len(comuns)

display(pd.Series(sobreposicoes, name='textos_sobrepostos').to_frame())
status_vazamento = 'PASS' if not any(sobreposicoes.values()) else 'BLOCKED'
(RELATORIOS / '08_vazamento.json').write_text(json.dumps({'passo': 8, 'status': status_vazamento, 'sobreposicoes': sobreposicoes}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert status_vazamento == 'PASS', 'Existe vazamento entre os conjuntos. Corrija a divisão antes de continuar.'

,textos_sobrepostos
treino_x_validacao,0
treino_x_avaliacao,0
validacao_x_avaliacao,0


## Passo 9 — Congelar o conjunto de avaliação

A avaliação será copiada para `dados/preparados` e receberá um hash. Depois deste ponto, ela não deve ser alterada durante os ajustes do treinamento.

In [26]:
avaliacao_congelada = PREPARADOS / 'avaliacao_congelada.jsonl'
avaliacao_congelada.write_text(''.join(json.dumps(registro, ensure_ascii=False) + '\n' for registro in avaliacao), encoding='utf-8')
manifesto_avaliacao = {
    'passo': 9,
    'arquivo': str(avaliacao_congelada),
    'sha256': sha256_arquivo(avaliacao_congelada),
    'registros': len(avaliacao),
    'congelado_em_utc': datetime.now(timezone.utc).isoformat(),
}
print(f"SHA-256: {manifesto_avaliacao['sha256']}")
(RELATORIOS / '09_avaliacao_congelada.json').write_text(json.dumps(manifesto_avaliacao, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

SHA-256: 5612ccd840999d21a60a87bff4279d51ddf511e4d1a16f4b97972742d409af8f


313

## Passo 10 — Converter para os formatos do LLaMA-Factory

O dataset mantém as duas estruturas usadas na aula: Alpaca e ShareGPT/OpenAI. A diferença é que o campo `output` do Alpaca e a mensagem `assistant` do formato conversacional recebem uma string com JSON válido.

Exemplo de resposta esperada: `{"sentimento":"positivo"}`.

In [27]:
def resposta_json(rotulo):
    return json.dumps({'sentimento': rotulo}, ensure_ascii=False, separators=(',', ':'))

def converter_para_alpaca(registros):
    return [{'instruction': registro['instruction'], 'input': registro['context'], 'output': resposta_json(registro['response'])} for registro in registros]

def converter_para_conversacional(registros):
    return [{'messages': [
        {'role': 'system', 'content': 'Você classifica sentimentos em textos em português e responde somente com JSON válido.'},
        {'role': 'user', 'content': registro['instruction'] + '\n\nTexto: ' + registro['context']},
        {'role': 'assistant', 'content': resposta_json(registro['response'])},
    ]} for registro in registros]

nomes_finais = {'treino': 'escutia_train.json', 'validacao': 'escutia_validation.json', 'avaliacao': 'escutia_evaluation.json'}
nomes_conversacionais = {'treino': 'escutia_train_conversacional.json', 'validacao': 'escutia_validation_conversacional.json', 'avaliacao': 'escutia_evaluation_conversacional.json'}
arquivos_finais, arquivos_conversacionais = {}, {}
for nome, registros in conjuntos.items():
    arquivo = PREPARADOS / nomes_finais[nome]
    arquivo.write_text(json.dumps(converter_para_alpaca(registros), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    arquivos_finais[nome] = arquivo
    arquivo_conversacional = PREPARADOS / nomes_conversacionais[nome]
    arquivo_conversacional.write_text(json.dumps(converter_para_conversacional(registros), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    arquivos_conversacionais[nome] = arquivo_conversacional

dataset_info = {}
for nome in nomes_finais:
    dataset_info[f'escutia_{nome}'] = {'file_name': nomes_finais[nome], 'columns': {'prompt': 'instruction', 'query': 'input', 'response': 'output'}}
    dataset_info[f'escutia_{nome}_conversacional'] = {'file_name': nomes_conversacionais[nome], 'formatting': 'sharegpt', 'columns': {'messages': 'messages'}, 'tags': {'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user', 'assistant_tag': 'assistant', 'system_tag': 'system'}}
(PREPARADOS / 'dataset_info.json').write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame(converter_para_alpaca(treino)).head())
display(pd.DataFrame(converter_para_conversacional(treino)).head())

,instruction,input,output
0,Classifique o sentimento predominante do texto...,"O Padre Fabio de Melo me passa uma paz, deve s...","{""sentimento"":""positivo""}"
1,Classifique o sentimento predominante do texto...,"#VideoShowAoVivo Marinho da Bahia ligado , pro...","{""sentimento"":""positivo""}"
2,Classifique o sentimento predominante do texto...,E eu que chorei com Maiara e Maraisa cantando ...,"{""sentimento"":""positivo""}"
3,Classifique o sentimento predominante do texto...,Irmãs Galvão que amorzinho! Minha avó gostava ...,"{""sentimento"":""positivo""}"
4,Classifique o sentimento predominante do texto...,"O meu Douglas saiu, eu não estou acreditando #...","{""sentimento"":""negativo""}"


,messages
0,"[{'role': 'system', 'content': 'Você classific..."
1,"[{'role': 'system', 'content': 'Você classific..."
2,"[{'role': 'system', 'content': 'Você classific..."
3,"[{'role': 'system', 'content': 'Você classific..."
4,"[{'role': 'system', 'content': 'Você classific..."


## Passo 11 — Validar o dataset final

A etapa final verifica schema, rótulos, resposta JSON, arquivos, isolamento entre conjuntos, registro `dataset_info.json` e existência da avaliação congelada.

A decisão `DATA_READY_FOR_SFT` significa somente que a preparação de dados foi concluída. Ainda será necessário revisar modelo, template, hardware, estratégia LoRA/QLoRA e autorização do treinamento.

In [28]:
schema_final = {
    'type': 'object',
    'required': ['instruction', 'input', 'output'],
    'additionalProperties': False,
    'properties': {'instruction': {'type': 'string', 'minLength': 1}, 'input': {'type': 'string', 'minLength': 1}, 'output': {'type': 'string', 'minLength': 1}},
}
validador = Draft202012Validator(schema_final)

def resposta_valida(valor):
    try:
        resposta = json.loads(valor)
        return set(resposta) == {'sentimento'} and resposta['sentimento'] in ROTULOS
    except (TypeError, json.JSONDecodeError):
        return False

resultados_finais, dados_finais = [], {}
for nome, arquivo in arquivos_finais.items():
    registros = json.loads(arquivo.read_text(encoding='utf-8'))
    dados_finais[nome] = registros
    erros_schema = [erro.message for registro in registros for erro in validador.iter_errors(registro)]
    erros_json = [indice for indice, registro in enumerate(registros) if not resposta_valida(registro.get('output'))]
    erros = erros_schema + [f'resposta_json_invalida:{indice}' for indice in erros_json]
    resultados_finais.append({'checagem': f'schema_{nome}', 'status': 'PASS' if registros and not erros else 'FAIL', 'registros': len(registros), 'erros': erros[:10]})

for nome, arquivo in arquivos_conversacionais.items():
    registros = json.loads(arquivo.read_text(encoding='utf-8'))
    erros = []
    for registro in registros:
        mensagens = registro.get('messages', [])
        if [mensagem.get('role') for mensagem in mensagens] != ['system', 'user', 'assistant'] or not resposta_valida(mensagens[-1].get('content', '')):
            erros.append(registro)
    resultados_finais.append({'checagem': f'conversacional_{nome}', 'status': 'PASS' if registros and not erros else 'FAIL', 'registros': len(registros), 'erros': len(erros)})

chaves_finais = {nome: {chave_comparacao(registro['input']) for registro in registros} for nome, registros in dados_finais.items()}
for esquerda, direita in [('treino', 'validacao'), ('treino', 'avaliacao'), ('validacao', 'avaliacao')]:
    comuns = chaves_finais[esquerda] & chaves_finais[direita]
    resultados_finais.append({'checagem': f'isolamento_{esquerda}_x_{direita}', 'status': 'PASS' if not comuns else 'FAIL', 'sobreposicoes': len(comuns)})

info = json.loads((PREPARADOS / 'dataset_info.json').read_text(encoding='utf-8'))
info_ok = all(info[f'escutia_{nome}']['file_name'] == nomes_finais[nome] and info[f'escutia_{nome}_conversacional']['file_name'] == nomes_conversacionais[nome] for nome in nomes_finais)
resultados_finais.append({'checagem': 'dataset_info', 'status': 'PASS' if info_ok else 'FAIL'})
resultados_finais.append({'checagem': 'avaliacao_congelada', 'status': 'PASS' if (PREPARADOS / 'avaliacao_congelada.jsonl').exists() else 'FAIL'})

pronto = all(resultado['status'] == 'PASS' for resultado in resultados_finais)
decisao = 'DATA_READY_FOR_SFT' if pronto else 'BLOQUEADO'
display(pd.DataFrame(resultados_finais))
relatorio_final = {'passo': 11, 'decisao': decisao, 'resultados': resultados_finais, 'distribuicao': {nome: dict(Counter(json.loads(registro['output'])['sentimento'] for registro in registros)) for nome, registros in dados_finais.items()}, 'gerado_em_utc': datetime.now(timezone.utc).isoformat()}
(RELATORIOS / '11_validacao_final.json').write_text(json.dumps(relatorio_final, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
manifesto = {'version': '2-json-sentimento', 'source_sha256': {'dataset.csv': sha256_arquivo(DATASET_UNIFICADO), 'dataset_local.csv': sha256_arquivo(LOCAL_DATASET)}, 'counts': {nome: len(registros) for nome, registros in dados_finais.items()}, 'training_authorized': pronto}
MANIFESTO.write_text(json.dumps(manifesto, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'DECISÃO: {decisao}')
print(f'ARQUIVO GERADO: {RELATORIOS / "11_validacao_final.json"}')
assert pronto, 'O dataset está bloqueado. Corrija os itens FAIL antes de avançar.'

,checagem,status,registros,erros,sobreposicoes
0,schema_treino,PASS,1833.0,[],NaN
1,schema_validacao,PASS,612.0,[],NaN
2,schema_avaliacao,PASS,612.0,[],NaN
3,conversacional_treino,PASS,1833.0,0,NaN
4,conversacional_validacao,PASS,612.0,0,NaN
5,conversacional_avaliacao,PASS,612.0,0,NaN
6,isolamento_treino_x_validacao,PASS,NaN,NaN,0.0
7,isolamento_treino_x_avaliacao,PASS,NaN,NaN,0.0
8,isolamento_validacao_x_avaliacao,PASS,NaN,NaN,0.0
9,dataset_info,PASS,NaN,NaN,NaN


DECISÃO: DATA_READY_FOR_SFT
ARQUIVO GERADO: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\dataset\dados\relatorios\11_validacao_final.json


## Checklist da Parte 1

Nesta parte, transformamos a fonte de sentimentos em português em um conjunto de dados preparado e controlado para o fine-tuning do EscutIA.

Normalizamos os textos, identificamos duplicidades, registramos alertas para revisão, criamos os conjuntos de treinamento, validação e avaliação, congelamos a avaliação e convertemos os dados para Alpaca e ShareGPT/OpenAI com resposta JSON.

Quando a validação final retorna `DATA_READY_FOR_SFT`, os dados estão prontos para a próxima etapa do projeto. Isso não inicia o treinamento.